# Hardware-Aware Hyperparameter Search for the Model Factory

I built this notebook as the main control panel for the hyperparameter-search framework in our repository. It lets me choose a model and dataset, define a large search space, calibrate the search against the hardware Colab actually gives me, estimate the time and compute required, run the search, resume it after a disconnect, and export the best hyperparameters for the final training and report notebook.

The notebook supports:

- **Proxy search** for quickly screening a large search space with smaller budgets.
- **Successive halving** for giving more resources only to the strongest candidates.
- **Full search** for training every sampled configuration at full fidelity.
- **Continuous execution** for repeating bounded search sessions until a trial, time, compute, accuracy, or stagnation limit is reached.
- Search spaces defined through a built-in CSV, my own CSV, a Python dictionary, a Python list, JSON, YAML, or direct manual input.
- Hardware-aware batch calibration, CPU-thread planning, precision selection, persistence, Pareto analysis, and cost accounting.

## How this connects to our final project

This notebook is not the final report by itself. Its purpose is to produce a defensible set of hyperparameters and the evidence behind that choice. The exported files can be used in the final project notebook to:

1. Train the selected ResNet and Vision Transformer configurations at full fidelity.
2. Compare model accuracy, training time, data use, memory use, and compute cost.
3. Show the Pareto frontier instead of presenting accuracy as the only definition of “best.”
4. Explain how the Model Factory adapts its search to the hardware available.
5. Preserve the final CIFAR test set until model selection is complete.

The same framework can later be adapted to our audio-language-transformer project by replacing the vision model and dataset adapters with speech-model and audio-dataset adapters. The HPO logic, persistence, resource budgets, Pareto selection, and compute accounting can remain mostly unchanged.


## Step-by-step workflow

I recommend following the notebook in order:

1. **Select a GPU runtime in Colab:** `Runtime → Change runtime type → T4 GPU` or the best accelerator available.
2. Run **Setup** to clone the HPO branch and install its additional dependencies.
3. Decide whether to save the study in **Google Drive**. I recommend Drive for any search I may need to resume.
4. Review the detected **hardware profile** and automatic resource plan.
5. Select the **model, dataset, and run label**.
6. Choose how I want to provide the **search space**.
7. Validate the search space before allocating training resources.
8. Select the **search mode, objectives, constraints, and budgets**.
9. Run batch and end-to-end **calibration**.
10. Review the **pre-search estimate**.
11. Change `START_SEARCH` to `True` only when the estimate and configuration look reasonable.
12. Analyze the completed candidates and export the selected hyperparameters.

A cell that can start an expensive job is disabled by default. This prevents `Run all` from accidentally launching a long search.


## 1. Setup


This cell clones the repository when it is not already present. It does not delete an existing checkout unless I explicitly set `FORCE_RECLONE=True`.

- Keep `BRANCH="feature/hpo-framework"` while working from the HPO branch.
- Change it to `main` after the branch is merged.
- When I change repository code, I can set `FORCE_RECLONE=True` once to start from a clean checkout.


In [ ]:
from pathlib import Path
from IPython.display import display
import copy
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/TrueRottweiler/WashingtonCsed504.git"
BRANCH = "feature/hpo-framework"
REPO_ROOT = Path("/content/WashingtonCsed504")
FORCE_RECLONE = False

if FORCE_RECLONE and REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

if not REPO_ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_ROOT),
        ],
        check=True,
    )
elif not (REPO_ROOT / "src/a1-cv/hpo").exists():
    raise RuntimeError(
        f"{REPO_ROOT} exists, but it does not contain the HPO package. "
        "Set FORCE_RECLONE=True or choose a different REPO_ROOT."
    )
else:
    print("Using the existing repository checkout:", REPO_ROOT)

CV_DIR = REPO_ROOT / "src/a1-cv"
os.chdir(CV_DIR)
if str(CV_DIR) not in sys.path:
    sys.path.insert(0, str(CV_DIR))

requirements = CV_DIR / "hpo_requirements.txt"
if not requirements.exists():
    raise FileNotFoundError(requirements)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
    check=True,
)

import optuna
import pandas as pd
import torch

revision = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print("Repository revision:", revision)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("Optuna:", optuna.__version__)


## 2. Persistence and Google Drive


The HPO framework saves its SQLite database, trial records, checkpoints, reports, and exported configurations as it runs. I use Google Drive for longer searches because the Colab runtime can disconnect.

- Keep the **same study name and configuration** when resuming an interrupted study.
- Change `RUN_LABEL` later when intentionally starting a different experiment.
- If I change the configuration but reuse the same study name, the framework will reject the resume because the configuration hash no longer matches. That protects me from mixing incompatible experiments.


In [ ]:
USE_GOOGLE_DRIVE = True
RESUME_EXISTING_STUDY = True

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if USE_GOOGLE_DRIVE and IN_COLAB:
    drive.mount("/content/drive")
    PERSIST_ROOT = Path("/content/drive/MyDrive/WashingtonCsed504-HPO")
else:
    PERSIST_ROOT = Path("/content/WashingtonCsed504-HPO")

PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistence root:", PERSIST_ROOT)
print("Resume existing study:", RESUME_EXISTING_STUDY)


## 3. Hardware profile and resource plan


This section detects the hardware instead of assuming that every Colab session has the same GPU. The scheduler defaults to one active training trial per GPU, divides CPU threads conservatively, and avoids unnecessary DataLoader workers when the dataset already lives on the accelerator.

The automatic settings are a safe starting point. I can override them when I have measured evidence that another setting is faster.


In [ ]:
from hpo.hardware import detect_hardware
from hpo.scheduler import plan_resources

DEVICE_OVERRIDE = "auto"       # auto, cuda, cpu, or mps
CONCURRENT_TRIALS_OVERRIDE = None
INTRAOP_THREADS_OVERRIDE = None
INTEROP_THREADS_OVERRIDE = None
WORKERS_OVERRIDE = None
MEMORY_RESERVE_GB = 1.5

hardware = detect_hardware()
resource_plan = plan_resources(
    hardware,
    device=DEVICE_OVERRIDE,
    requested_concurrency=CONCURRENT_TRIALS_OVERRIDE,
    requested_intraop_threads=INTRAOP_THREADS_OVERRIDE,
    requested_interop_threads=INTEROP_THREADS_OVERRIDE,
    requested_workers=WORKERS_OVERRIDE,
    memory_reserve_gb=MEMORY_RESERVE_GB,
)

RESOLVED_DEVICE = resource_plan.device
if RESOLVED_DEVICE == "cuda":
    SELECTED_PRECISION = "bf16" if hardware.bf16_native else "fp16"
else:
    SELECTED_PRECISION = "fp32"

print("Hardware profile:")
print(json.dumps(hardware.to_dict(), indent=2, default=str))
print("\nResource plan:")
print(json.dumps(resource_plan.to_dict(), indent=2, default=str))
print("\nSelected precision:", SELECTED_PRECISION)


## 4. Experiment selection and run identity


I use the run label to separate experiments. I keep the same label when resuming and change it when I intentionally change the model, search space, objectives, or budgets.

The main tested workflows are CIFAR-10 and CIFAR-100 with ResNet or ViT. ImageNet-32 requires the prepared files in the repository’s expected `src/a1-cv/data/imagenet32` directory because the current repository loader uses that fixed location.


In [ ]:
DATASET = "cifar10"       # cifar10, cifar100, imagenet32
MODEL = "resnet18"        # resnet18, resnet50, vit, vit_base
RUN_LABEL = "run01"
SEARCH_SEED = 42
SPLIT_SEED = 42
VALIDATION_FRACTION = 0.10

SUPPORTED_DATASETS = {"cifar10", "cifar100", "imagenet32"}
SUPPORTED_MODELS = {"resnet18", "resnet50", "vit", "vit_base"}
if DATASET not in SUPPORTED_DATASETS:
    raise ValueError(f"Unsupported DATASET: {DATASET!r}")
if MODEL not in SUPPORTED_MODELS:
    raise ValueError(f"Unsupported MODEL: {MODEL!r}")

if DATASET == "imagenet32":
    expected_imagenet32 = CV_DIR / "data/imagenet32"
    required_files = [
        expected_imagenet32 / "train_x.npy",
        expected_imagenet32 / "train_y.npy",
        expected_imagenet32 / "val_x.npy",
        expected_imagenet32 / "val_y.npy",
        expected_imagenet32 / "stats.json",
    ]
    missing = [str(path) for path in required_files if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "ImageNet-32 is selected, but the prepared repository files are missing:\n"
            + "\n".join(missing)
            + "\nRun imagenet_prepare.py or place the prepared files in the expected directory."
        )

NUM_CLASSES = {"cifar10": 10, "cifar100": 100, "imagenet32": 1000}[DATASET]
print({"dataset": DATASET, "model": MODEL, "classes": NUM_CLASSES, "run_label": RUN_LABEL})


## 5. Search-space source


I can provide the search space in several ways. Every input is normalized into the same internal schema before the search begins.

- `builtin`: use the repository CSV for the selected model family.
- `dictionary`: edit `DICTIONARY_SPACE` directly.
- `list`: edit `LIST_SPACE` directly.
- `csv_upload`: upload one CSV through Colab.
- `csv`, `json`, or `yaml`: provide a path in `SEARCH_SPACE_FILE`.
- `manual`: use the editable manual dictionary.

The CSV format uses columns such as `name`, `type`, `low`, `high`, `choices`, `step`, `log`, `default`, `condition`, and `enabled`.


In [ ]:
from hpo.notebook_api import normalize_uploaded_csv, preview_dataframe
from hpo.search_space import combination_count, load_csv, load_space, normalize_space

INPUT_SOURCE = "builtin"
SEARCH_SPACE_FILE = Path("/content/search_space.csv")

BUILTIN_CSV = CV_DIR / (
    "hpo_configs/search_spaces/vit_cifar.csv"
    if MODEL.startswith("vit")
    else "hpo_configs/search_spaces/resnet18_cifar.csv"
)

DICTIONARY_SPACE = {
    "learning_rate": {
        "type": "float",
        "low": 1e-4,
        "high": 0.2,
        "log": True,
        "default": 0.01,
    },
    "batch_size": {
        "type": "categorical",
        "choices": [64, 128, 256],
        "default": 128,
    },
    "optimizer": {
        "type": "categorical",
        "choices": ["sgd", "adamw"],
        "default": "sgd",
    },
    "momentum": {
        "type": "float",
        "low": 0.80,
        "high": 0.95,
        "step": 0.05,
        "default": 0.90,
        "condition": 'optimizer == "sgd"',
    },
    "beta1": {
        "type": "float",
        "low": 0.80,
        "high": 0.95,
        "step": 0.05,
        "default": 0.90,
        "condition": 'optimizer == "adamw"',
    },
}

LIST_SPACE = [
    {"name": name, **definition}
    for name, definition in DICTIONARY_SPACE.items()
]
MANUAL_SPACE = copy.deepcopy(DICTIONARY_SPACE)

if INPUT_SOURCE == "builtin":
    specs = load_csv(BUILTIN_CSV)
elif INPUT_SOURCE == "dictionary":
    specs = normalize_space(DICTIONARY_SPACE, source_name="dictionary cell")
elif INPUT_SOURCE == "list":
    specs = normalize_space(LIST_SPACE, source_name="list cell")
elif INPUT_SOURCE == "manual":
    specs = normalize_space(MANUAL_SPACE, source_name="manual cell")
elif INPUT_SOURCE == "csv_upload":
    if not IN_COLAB:
        raise RuntimeError("csv_upload is available in Colab. Use INPUT_SOURCE='csv' outside Colab.")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV search-space file.")
    filename, content = next(iter(uploaded.items()))
    specs = normalize_uploaded_csv(content, filename=filename)
elif INPUT_SOURCE in {"csv", "json", "yaml"}:
    specs = load_space(SEARCH_SPACE_FILE)
else:
    raise ValueError(f"Unsupported INPUT_SOURCE: {INPUT_SOURCE!r}")

if not specs:
    raise ValueError("The normalized search space is empty.")

display(preview_dataframe(specs))
finite_combinations = combination_count(specs)
print("Input source:", INPUT_SOURCE)
print("Finite combination count:", finite_combinations if finite_combinations is not None else "continuous or too large")


## 6. Validate the search space before training


The original notebook displayed the conditions and defaults but did not actually pass a representative candidate through the repository’s model and optimizer constraints. This version builds an active default candidate, respects conditional fields, and validates it before any expensive work begins.


In [ ]:
import math
from hpo.conditions import SafeCondition
from hpo.constraints import validate_candidate


def representative_value(spec):
    if spec.default is not None:
        return spec.default
    if spec.type in {"categorical", "bool"}:
        return spec.choices[0]
    if spec.type == "fixed":
        return spec.default
    if spec.type == "int":
        return int((int(spec.low) + int(spec.high)) // 2)
    if spec.type == "float":
        if spec.log:
            return math.sqrt(float(spec.low) * float(spec.high))
        return (float(spec.low) + float(spec.high)) / 2
    raise ValueError(spec.type)


def build_representative_candidate(parameter_specs):
    values = {}
    pending = list(parameter_specs)
    while pending:
        progressed = False
        for spec in list(pending):
            condition = SafeCondition(spec.condition) if spec.condition else None
            if condition and not condition.references.issubset(values):
                continue
            pending.remove(spec)
            progressed = True
            if condition and not condition.evaluate(values):
                continue
            values[spec.name] = representative_value(spec)
        if not progressed:
            raise RuntimeError("The conditional search-space graph could not be resolved.")
    return values

representative_params = build_representative_candidate(specs)
runtime_preview = {
    "device": RESOLVED_DEVICE,
    "precision": SELECTED_PRECISION,
    "channels_last": MODEL.startswith("resnet") and RESOLVED_DEVICE == "cuda",
}
validate_candidate(representative_params, {"name": MODEL}, runtime_preview)

print("Representative candidate passed structural validation:")
print(json.dumps(representative_params, indent=2, default=str))
print("Conditional parameters:", [spec.condition for spec in specs if spec.condition])


## 7. Search mode, budgets, objectives, constraints, and costs


### Choosing a mode

- **Proxy:** best for checking a very large space quickly. It uses a smaller data or epoch budget and stops after the screening stage.
- **Successive halving:** my recommended default. It screens many candidates, promotes the strongest candidates through larger budgets, and optionally performs full-fidelity confirmation.
- **Full:** every valid sampled candidate receives the full budget. This is the most expensive option.
- **Continuous:** this is a resumable wrapper around one of the three modes. It keeps running bounded sessions until a stopping rule is reached.

For a first Colab run, I recommend successive halving with the practical defaults below. I can increase the trial and epoch budgets after the smoke notebook and estimate look correct.


In [ ]:
MODE = "successive_halving"       # proxy, successive_halving, full
CONTINUOUS = False

# Proxy budget
PROXY_TRIALS = 8
PROXY_EPOCHS = 2
PROXY_MAX_STEPS = 300
PROXY_DATA_FRACTION = 0.25
PROXY_VALIDATION_FRACTION = 0.50
PROXY_PROMOTE_TOP_K = 4

# Successive-halving budget
HALVING_RESOURCE_TYPE = "epochs"
HALVING_RUNGS = [5, 12]
HALVING_REDUCTION_FACTOR = 2
PROMOTE_TO_FULL = True

# Full-fidelity budget
FULL_TRIALS = 2
FULL_EPOCHS = 30 if MODEL.startswith("resnet") else 80
FULL_MAX_STEPS = None
FULL_SEEDS = [42]

CONTINUOUS_SETTINGS = {
    "enabled": CONTINUOUS,
    "strategy": MODE,
    "maximum_trials": 32 if CONTINUOUS else None,
    "maximum_wall_time_hours": 8.0 if CONTINUOUS else None,
    "maximum_gpu_hours": None,
    "maximum_cpu_hours": None,
    "maximum_cost_usd": None,
    "target_validation_metric": None,
    "stop_after_no_improvement_trials": 25,
    "minimum_improvement": 0.0005,
    "pareto_stagnation_trials": 25,
    "checkpoint_after_each_trial": True,
}

if MODE not in {"proxy", "successive_halving", "full"}:
    raise ValueError(MODE)
if CONTINUOUS and not any(
    CONTINUOUS_SETTINGS[name] is not None
    for name in (
        "maximum_trials",
        "maximum_wall_time_hours",
        "maximum_gpu_hours",
        "maximum_cpu_hours",
        "maximum_cost_usd",
        "target_validation_metric",
        "stop_after_no_improvement_trials",
        "pareto_stagnation_trials",
    )
):
    raise ValueError("Continuous execution requires at least one stopping rule.")

OBJECTIVE_PROFILE = "pareto_accuracy_time_memory"
SINGLE_OBJECTIVE_SAMPLER = "tpe"  # tpe, random, qmc

if OBJECTIVE_PROFILE == "accuracy_only":
    OBJECTIVES = [
        {"name": "validation_top1", "direction": "maximize", "primary": True}
    ]
    EFFECTIVE_SAMPLER = SINGLE_OBJECTIVE_SAMPLER
    OBJECTIVE_TAG = "accuracy"
elif OBJECTIVE_PROFILE == "pareto_accuracy_time":
    OBJECTIVES = [
        {"name": "validation_top1", "direction": "maximize", "primary": True},
        {"name": "wall_seconds", "direction": "minimize", "primary": False},
    ]
    EFFECTIVE_SAMPLER = "nsga2"
    OBJECTIVE_TAG = "pareto-accuracy-time"
elif OBJECTIVE_PROFILE == "pareto_accuracy_time_memory":
    OBJECTIVES = [
        {"name": "validation_top1", "direction": "maximize", "primary": True},
        {"name": "wall_seconds", "direction": "minimize", "primary": False},
        {"name": "peak_gpu_memory_mb", "direction": "minimize", "primary": False},
    ]
    EFFECTIVE_SAMPLER = "nsga2"
    OBJECTIVE_TAG = "pareto-accuracy-time-memory"
else:
    raise ValueError(OBJECTIVE_PROFILE)

# Derive the memory constraint from the assigned accelerator instead of assuming a 12 or 14 GB GPU.
CONSTRAINTS = []
if RESOLVED_DEVICE == "cuda" and hardware.gpus:
    selected_gpu = hardware.gpus[0]
    detected_vram_gb = selected_gpu.available_vram_gb or selected_gpu.total_vram_gb
    usable_vram_mb = max(512, int((detected_vram_gb - MEMORY_RESERVE_GB) * 1024))
    CONSTRAINTS.append(
        {"name": "peak_gpu_memory_mb", "operator": "<=", "value": usable_vram_mb}
    )

# Monetary estimates require user-supplied marginal rates. Compute quantities are still recorded when rates are absent.
GPU_USD_PER_HOUR = None
CPU_USD_PER_HOUR = None
STORAGE_USD_PER_GB_MONTH = None
COLAB_SUBSCRIPTION_USD = None
COLAB_COMPUTE_UNIT_USD = None

COST_RATES = {
    "gpu_usd_per_hour": GPU_USD_PER_HOUR,
    "cpu_usd_per_hour": CPU_USD_PER_HOUR,
    "storage_usd_per_gb_month": STORAGE_USD_PER_GB_MONTH,
    "electricity_usd_per_kwh": None,
    "colab_subscription_usd": COLAB_SUBSCRIPTION_USD,
    "colab_compute_unit_usd": COLAB_COMPUTE_UNIT_USD,
}
REQUIRED_COST_FIELDS = (
    "gpu_usd_per_hour",
    "cpu_usd_per_hour",
    "storage_usd_per_gb_month",
)
MISSING_COST_RATES = [name for name in REQUIRED_COST_FIELDS if COST_RATES[name] is None]
COST_ESTIMATION_AVAILABLE = not MISSING_COST_RATES

print(json.dumps({
    "mode": MODE,
    "continuous": CONTINUOUS_SETTINGS,
    "objectives": OBJECTIVES,
    "sampler": EFFECTIVE_SAMPLER,
    "constraints": CONSTRAINTS,
    "cost_estimation_available": COST_ESTIMATION_AVAILABLE,
    "missing_cost_rates": MISSING_COST_RATES,
}, indent=2))


## 8. Batch-size calibration


The batch-size probe measures actual forward, backward, optimizer, throughput, and memory behavior on the selected hardware. I use it to remove batch sizes that fail or leave too little memory headroom.

The largest batch that fits is not automatically the best batch. The framework separately records the batch with the highest measured throughput.


In [ ]:
from dataclasses import replace
from hpo.adapters import RepoModules, build_trial_dataset, build_trial_model
from hpo.calibration import calibrate_batch_sizes

modules = RepoModules(REPO_ROOT)
device = torch.device(RESOLVED_DEVICE)
dataset_config = {
    "name": DATASET,
    "validation_fraction": VALIDATION_FRACTION,
    "split_seed": SPLIT_SEED,
    "strategy": "gpu_resident" if RESOLVED_DEVICE != "cpu" else "resident",
}
runtime_for_dataset = {
    "seed": SEARCH_SEED,
    "data_strategy": "auto",
    "workers": resource_plan.workers_per_trial,
}
bundle = build_trial_dataset(modules, dataset_config, runtime_for_dataset, device)
model_config = {"name": MODEL}

batch_spec = next((spec for spec in specs if spec.name == "batch_size"), None)
if batch_spec is not None and batch_spec.type == "categorical":
    calibration_candidates = [int(value) for value in batch_spec.choices]
else:
    calibration_candidates = [32, 64, 128, 256, 512]

# A fresh iterator is created for each candidate batch.
def make_batch(batch_size):
    return next(bundle.train.epoch(batch_size, train=True))

calibration = calibrate_batch_sizes(
    lambda: build_trial_model(modules, model_config, bundle.num_classes, device),
    make_batch,
    device=device,
    candidates=calibration_candidates,
    precision=SELECTED_PRECISION,
    warmup_steps=2,
    measure_steps=3,
    channels_last=MODEL.startswith("resnet") and RESOLVED_DEVICE == "cuda",
)
print(json.dumps(calibration.to_dict(), indent=2, default=str))

completed_batches = {
    measurement.batch_size
    for measurement in calibration.measurements
    if measurement.status == "completed"
}
if not completed_batches:
    raise RuntimeError("No candidate batch completed calibration on this hardware.")

if batch_spec is not None and batch_spec.type == "categorical":
    recommended_batches = [
        int(value)
        for value in batch_spec.choices
        if int(value) in calibration.recommended_candidates
        and int(value) in completed_batches
    ]
    fitting_batches = [
        int(value) for value in batch_spec.choices if int(value) in completed_batches
    ]
    safe_batch_choices = recommended_batches or fitting_batches
    preferred_default = (
        calibration.highest_throughput_batch
        if calibration.highest_throughput_batch in safe_batch_choices
        else safe_batch_choices[-1]
    )
    specs = [
        replace(spec, choices=tuple(safe_batch_choices), default=preferred_default)
        if spec.name == "batch_size"
        else spec
        for spec in specs
    ]
else:
    preferred_default = calibration.highest_throughput_batch
    safe_batch_choices = sorted(completed_batches)

print("Safe batch choices:", safe_batch_choices)
print("Pilot/default batch:", preferred_default)


## 9. Build and validate the resolved study configuration


This cell creates the exact YAML configuration used by the CLI and records it beside the study. The template is selected from the dataset and model family instead of always starting from the CIFAR-10 ResNet configuration.

I keep the same `RUN_LABEL` when resuming. I change it when I want a clean experiment with different settings.


In [ ]:
import yaml
from dataclasses import asdict
from hpo.config import load_study_config


def choose_template(model_name, dataset_name):
    if model_name.startswith("vit"):
        filename = "vit_cifar100.yaml" if dataset_name == "cifar100" else "vit_cifar10.yaml"
    else:
        filename = "resnet18_cifar100.yaml" if dataset_name == "cifar100" else "resnet18_cifar10_successive_halving.yaml"
    return CV_DIR / "hpo_configs/colab" / filename

TEMPLATE_PATH = choose_template(MODEL, DATASET)
if DATASET == "imagenet32":
    print(
        "ImageNet-32 currently reuses the closest architecture template. "
        "Review all budgets and the search space before starting the search."
    )

base = yaml.safe_load(TEMPLATE_PATH.read_text(encoding="utf-8"))
EXECUTION_TAG = "continuous" if CONTINUOUS else "bounded"
STUDY_NAME = (
    f"{MODEL}-{DATASET}-{MODE}-{EXECUTION_TAG}-"
    f"{OBJECTIVE_TAG}-{EFFECTIVE_SAMPLER}-s{SEARCH_SEED}-{RUN_LABEL}"
)
CONFIG_PATH = PERSIST_ROOT / f"{STUDY_NAME}.yaml"

base["study"].update({
    "name": STUDY_NAME,
    "output_dir": str(PERSIST_ROOT),
    "storage_path": str(PERSIST_ROOT / f"{STUDY_NAME}.db"),
    "resume": RESUME_EXISTING_STUDY,
    "seed": SEARCH_SEED,
})
base.setdefault("search", {})["mode"] = MODE
base["search"]["sampler"] = EFFECTIVE_SAMPLER
base["dataset"] = dataset_config
base["model"] = {"name": MODEL}
base["objectives"] = OBJECTIVES
base["constraints"] = CONSTRAINTS
base["cost_rates"] = COST_RATES
base["continuous"] = CONTINUOUS_SETTINGS

base["runtime"].update({
    "device": RESOLVED_DEVICE,
    "precision": SELECTED_PRECISION,
    "tf32": bool(RESOLVED_DEVICE == "cuda" and hardware.tf32_supported),
    "channels_last": bool(MODEL.startswith("resnet") and RESOLVED_DEVICE == "cuda"),
    "concurrent_trials": resource_plan.concurrent_trials,
    "intraop_threads": resource_plan.intraop_threads,
    "interop_threads": resource_plan.interop_threads,
    "workers": resource_plan.workers_per_trial,
    "memory_reserve_gb": MEMORY_RESERVE_GB,
})

base["proxy"].update({
    "enabled": MODE in {"proxy", "successive_halving"},
    "trials": PROXY_TRIALS,
    "budget": {
        "epochs": PROXY_EPOCHS,
        "max_steps": PROXY_MAX_STEPS,
        "data_fraction": PROXY_DATA_FRACTION,
        "validation_fraction": PROXY_VALIDATION_FRACTION,
        "validation_interval": 1,
        "seeds": [SEARCH_SEED],
    },
    "promote_top_k": PROXY_PROMOTE_TOP_K,
})
base["successive_halving"].update({
    "enabled": MODE == "successive_halving",
    "resource_type": HALVING_RESOURCE_TYPE,
    "rung_budgets": HALVING_RUNGS,
    "reduction_factor": HALVING_REDUCTION_FACTOR,
    "minimum_trials_per_rung": 1,
    "continue_checkpoints": True,
    "promotion_metric": "validation_top1",
})
base["full"].update({
    "enabled": MODE == "full" or (MODE == "successive_halving" and PROMOTE_TO_FULL),
    "trials": FULL_TRIALS,
    "budget": {
        "epochs": FULL_EPOCHS,
        "max_steps": FULL_MAX_STEPS,
        "data_fraction": 1.0,
        "validation_fraction": 1.0,
        "validation_interval": 1,
        "seeds": FULL_SEEDS,
    },
    "exhaustive": False,
    "maximum_combinations": 500,
    "allow_performance_pruning": False,
    "allow_safety_termination": True,
    "evaluate_test": False,
})

base["search_space"] = {
    spec.name: {
        key: value
        for key, value in spec.to_dict().items()
        if key not in {"name", "source", "item"} and value not in (None, [], ())
    }
    for spec in specs
}

CONFIG_PATH.write_text(yaml.safe_dump(base, sort_keys=False), encoding="utf-8")
config = load_study_config(CONFIG_PATH)

assert config.mode == MODE
assert config.sampler == EFFECTIVE_SAMPLER
assert config.continuous.enabled is CONTINUOUS
assert config.dataset.get("split_seed") == SPLIT_SEED
assert config.seed == SEARCH_SEED
assert not config.full.evaluate_test, "The test set must remain reserved during HPO."
validate_candidate(build_representative_candidate(config.search_space), config.model, asdict(config.runtime))

print("Resolved configuration:")
print(json.dumps({
    "study_name": config.name,
    "template": str(TEMPLATE_PATH),
    "mode": config.mode,
    "continuous": config.continuous.enabled,
    "sampler": config.sampler,
    "objectives": [asdict(item) for item in config.objectives],
    "device": config.runtime.device,
    "precision": config.runtime.precision,
    "concurrent_trials": config.runtime.concurrent_trials,
    "proxy_trials": config.proxy.trials,
    "proxy_budget": asdict(config.proxy.budget),
    "halving_rungs": config.successive_halving.rung_budgets,
    "full_trials": config.full.trials,
    "full_budget": asdict(config.full.budget),
    "config_path": str(CONFIG_PATH),
}, indent=2, default=str))


## 10. End-to-end calibration and pre-search estimate


The previous notebook estimated time from the batch-step probe alone and set validation and checkpoint overhead to zero. That made the estimate useful as a rough lower bound, but it did not represent the complete trial path.

This version also runs a short **end-to-end pilot trial** through the same `TrialRunner` used by HPO. It measures training time, validation, checkpointing, CPU time, memory, examples processed, and FLOP metadata. The estimator then uses the measured training rate plus the remaining per-epoch overhead.

The estimate is still a projection. Different sampled architectures, batch sizes, augmentations, and optimizers can change runtime. It becomes more reliable after I compare it with completed trials on the same hardware.


In [ ]:
from hpo.estimation import estimate_continuous_capacity, estimate_search
from hpo.trial_runner import TrialResource, TrialRunner

PILOT_STEPS = 10
PILOT_VALIDATION_FRACTION = 0.10
pilot_params = build_representative_candidate(config.search_space)
pilot_params["batch_size"] = int(preferred_default)
validate_candidate(pilot_params, config.model, asdict(config.runtime))

pilot_runner = TrialRunner(
    REPO_ROOT,
    dataset_config=config.dataset,
    model_config=config.model,
    runtime_config={**asdict(config.runtime), "seed": SEARCH_SEED},
    output_dir=PERSIST_ROOT / f"{STUDY_NAME}-calibration",
    hardware=hardware,
)
pilot_train_examples = pilot_runner.dataset.train_examples
pilot_data_fraction = min(
    1.0,
    PILOT_STEPS * pilot_params["batch_size"] / max(1, pilot_train_examples),
)
pilot_resource = TrialResource(
    stage="calibration",
    target_epochs=1,
    max_steps=PILOT_STEPS,
    data_fraction=pilot_data_fraction,
    validation_fraction=PILOT_VALIDATION_FRACTION,
    seed=SEARCH_SEED,
    continue_checkpoint=False,
)
pilot_result = pilot_runner.run(
    candidate_id="notebook-calibration",
    trial_number=None,
    params=pilot_params,
    resource=pilot_resource,
    maximum_total_epochs=1,
)
if pilot_result.status != "completed":
    raise RuntimeError(
        "End-to-end calibration failed: "
        + str(pilot_result.failure_reason or pilot_result.invalid_reason)
    )

pilot_metrics = pilot_result.metrics
training_seconds = sum(
    float(epoch.get("train", {}).get("sec", 0.0))
    for epoch in pilot_metrics.get("history", [])
)
non_training_overhead = max(0.0, float(pilot_metrics["wall_seconds"]) - training_seconds)
calibration_record = {
    "batch_size": pilot_params["batch_size"],
    "seconds_per_example": training_seconds / max(1, pilot_metrics["training_examples"]),
    "evaluation_seconds_per_epoch": non_training_overhead,
    "checkpoint_seconds_per_epoch": 0.0,
    "checkpoint_size_mb": pilot_metrics.get("checkpoint_size_mb", 0.0),
    "peak_memory_mb": pilot_metrics.get("peak_gpu_memory_mb", 0.0),
    "train_flops_per_image": pilot_metrics.get("train_flops_per_image"),
    "cpu_seconds": pilot_metrics.get("cpu_seconds", 0.0),
    "elapsed_seconds": pilot_metrics.get("wall_seconds", 0.0),
}

estimate_arguments = {
    "train_examples": pilot_runner.dataset.train_examples,
    "validation_examples": pilot_runner.dataset.validation_examples,
    "calibration_records": [calibration_record],
    "representative_batch_size": pilot_params["batch_size"],
}
if config.continuous.enabled:
    estimate_payload = estimate_continuous_capacity(config, **estimate_arguments)
else:
    estimate_payload = estimate_search(config, **estimate_arguments).to_dict()

# The current estimator reports total serial work. This extra field gives a clearly labeled
# heuristic makespan for independent trials across multiple devices; it is not used as a measurement.
if not config.continuous.enabled and resource_plan.concurrent_trials > 1:
    expected_serial = estimate_payload["duration_seconds"]["expected"]
    estimate_payload["heuristic_parallel_makespan_seconds"] = (
        expected_serial / resource_plan.concurrent_trials
    )
    estimate_payload.setdefault("assumptions", []).append(
        "parallel makespan divides serial work by trial concurrency and does not model scaling loss"
    )

ESTIMATE_PATH = PERSIST_ROOT / f"{STUDY_NAME}-pre_search_estimate.json"
ESTIMATE_PATH.write_text(json.dumps(estimate_payload, indent=2), encoding="utf-8")

print("End-to-end pilot result:")
print(json.dumps({
    "status": pilot_result.status,
    "params": pilot_params,
    "training_seconds": training_seconds,
    "non_training_overhead_seconds": non_training_overhead,
    "metrics": {key: value for key, value in pilot_metrics.items() if key != "history"},
}, indent=2, default=str))
print("\nPre-search estimate:")
print(json.dumps(estimate_payload, indent=2, default=str))
print("\nEstimate saved to:", ESTIMATE_PATH)


## 11. How to read the estimate


Before I start the search, I check the following:

- **Expected trials:** how many initial candidates the selected mode evaluates.
- **Pruned and promoted trials:** how the proxy or successive-halving stages reduce the workload.
- **Full evaluations:** how many candidates receive the complete budget.
- **Expected examples and epochs:** the estimated data consumed across all stages.
- **Duration range:** optimistic, expected, and conservative projections.
- **GPU-hours and CPU-hours:** compute consumption, which is different from elapsed wall-clock time when trials run in parallel.
- **Peak memory:** measured during calibration; other sampled models may use more or less.
- **Monetary cost:** only available when I provide the required hourly and storage rates.

For continuous mode, the output estimates how much work fits inside the configured budget. It does not claim that an open-ended search has a fixed completion time.


## 12. Search execution


This is the expensive launch gate. I leave `START_SEARCH=False` while reviewing or submitting the notebook. To run the search, I change it to `True` and execute this cell.

The search runs in a subprocess so the notebook can monitor progress and preserve a complete log. Pressing Stop or interrupting the cell terminates the subprocess; completed trials remain saved.


In [ ]:
from hpo.monitoring import run_command_with_monitor

START_SEARCH = False

if START_SEARCH:
    study_dir = PERSIST_ROOT / STUDY_NAME
    log_path = PERSIST_ROOT / f"{STUDY_NAME}.log"
    command = [
        sys.executable,
        "-m",
        "hpo.cli",
        "--repo-root",
        str(REPO_ROOT),
        "search",
        "--config",
        str(CONFIG_PATH),
        "--mode",
        MODE,
    ]
    if CONTINUOUS:
        command.append("--continuous")

    # The monitor uses this only for a rough remaining-work display.
    expected_records = None
    if not CONTINUOUS:
        if MODE == "proxy":
            expected_records = config.proxy.trials
        elif MODE == "full":
            expected_records = config.full.trials * (len(config.full.budget.seeds) + 1)

    return_code = run_command_with_monitor(
        command,
        cwd=CV_DIR,
        study_dir=study_dir,
        log_path=log_path,
        interval_seconds=15,
        timeout_seconds=None,
        expected_records=expected_records,
    )
    print("Return code:", return_code)
    if return_code != 0:
        tail = log_path.read_text(encoding="utf-8").splitlines()[-120:]
        print("\n".join(tail))
        raise RuntimeError("The search failed. Review the log tail above.")
else:
    print("Search not started. Review the configuration and estimate, then set START_SEARCH=True.")


## 13. Resume after a Colab disconnect


To resume:

1. Reconnect to Colab and select the accelerator again.
2. Rerun Setup, Persistence, Hardware, Experiment, Search Space, Search Mode, Calibration, and Configuration cells.
3. Keep the same `RUN_LABEL`, seeds, search space, budgets, objectives, and study name.
4. Set `RESUME_SEARCH=True` below.

The framework reloads the SQLite study, state files, sampler state, and checkpoints. It should not repeat completed candidates.


In [ ]:
RESUME_SEARCH = False

if RESUME_SEARCH:
    from hpo.study import HpoStudy
    resumed_summary = HpoStudy(CONFIG_PATH, repo_root=REPO_ROOT).run()
    print(json.dumps(resumed_summary, indent=2, default=str))
else:
    print("Resume is disabled. Completed trials remain stored under:", PERSIST_ROOT / STUDY_NAME)


## 14. Results and Pareto selection


The original analysis cell selected from every completed record, which could mix proxy or intermediate-rung results with full-fidelity candidates. This version keeps the highest-fidelity completed result for each candidate before applying the selection rules.

“Best” means best observed for this validation split, search space, mode, fidelity, seeds, objectives, constraints, and compute budget. It does not mean a universal global optimum.


In [ ]:
from hpo.costing import estimate_cost
from hpo.persistence import read_jsonl
from hpo.reporting import export_reports
from hpo.schemas import ObjectiveSpec
from hpo.selection import (
    fastest_above_accuracy,
    lowest_cost_within_accuracy_margin,
    lowest_memory_above_accuracy,
    pareto_knee,
)

MINIMUM_ACCEPTABLE_ACCURACY = {
    "cifar10": 0.80,
    "cifar100": 0.55,
    "imagenet32": 0.25,
}[DATASET]  # Adjust this threshold for the experiment and budget.
study_dir = PERSIST_ROOT / STUDY_NAME
selected_candidate = None

if (study_dir / "trials.jsonl").exists():
    rows = read_jsonl(study_dir / "trials.jsonl")
    objectives = [ObjectiveSpec(**objective) for objective in OBJECTIVES]
    report = export_reports(study_dir, objectives)

    stage_order = {"proxy": 0, "halving": 1, "full": 2}
    latest_completed = {}
    for row in rows:
        if row.get("status") != "completed" or not row.get("metrics"):
            continue
        candidate_id = str(row.get("candidate_id"))
        current = latest_completed.get(candidate_id)
        if current is None or stage_order.get(row.get("stage"), -1) >= stage_order.get(current.get("stage"), -1):
            latest_completed[candidate_id] = row
    final_rows = list(latest_completed.values())

    print(json.dumps(report, indent=2, default=str))
    if (study_dir / "completed_candidates.csv").exists():
        display(pd.read_csv(study_dir / "completed_candidates.csv"))
    if (study_dir / "pareto_trials.csv").exists():
        display(pd.read_csv(study_dir / "pareto_trials.csv"))

    fastest = fastest_above_accuracy(final_rows, minimum_accuracy=MINIMUM_ACCEPTABLE_ACCURACY)
    lowest_memory = lowest_memory_above_accuracy(final_rows, minimum_accuracy=MINIMUM_ACCEPTABLE_ACCURACY)
    knee = pareto_knee(final_rows, objectives)

    print("Fastest acceptable candidate:", fastest)
    print("Lowest-memory acceptable candidate:", lowest_memory)
    print("Pareto knee candidate:", knee)

    if COST_ESTIMATION_AVAILABLE:
        cost_adjusted_rows = copy.deepcopy(final_rows)
        for row in cost_adjusted_rows:
            row["metrics"].update(estimate_cost(row["metrics"], config.cost_rates))
        print(
            "Lowest-cost candidate near the best accuracy:",
            lowest_cost_within_accuracy_margin(cost_adjusted_rows, accuracy_margin=0.01),
        )
    else:
        print("Cost-based selection skipped. Missing rates:", MISSING_COST_RATES)

    if knee is not None:
        selected_candidate = knee
    elif final_rows:
        selected_candidate = max(
            final_rows,
            key=lambda row: float(row["metrics"].get("validation_top1", float("-inf"))),
        )
else:
    print("No study results exist yet. Run or resume the search first.")


## 15. Benchmark against the repository reference configuration


The stored repository results provide useful historical context, but they are not automatically a fair comparison because they may use a different split, seed, hardware, epoch budget, or test-set protocol.

For a defensible comparison, this section creates a gated **same-protocol reference configuration** using the same validation split, seeds, runtime settings, and full-fidelity budget as the discovery study. I only enable `RUN_REFERENCE_TRACK` after reviewing its estimated cost.


In [ ]:
from hpo.baselines import REPOSITORY_RECIPES, load_repository_baselines
from hpo.benchmark import normalized_parameter_distance
from hpo.study import HpoStudy

RUN_REFERENCE_TRACK = False
REFERENCE_CONFIG_PATH = None
FAIR_COMPARISON_PATH = None

historical_baselines = [
    baseline
    for baseline in load_repository_baselines(REPO_ROOT)
    if baseline.dataset == DATASET and baseline.model == MODEL
]
historical_reference = (
    max(
        historical_baselines,
        key=lambda baseline: baseline.validation_top1 if baseline.validation_top1 is not None else float("-inf"),
    )
    if historical_baselines
    else None
)
if historical_reference is not None:
    print("Historical context only:")
    print(json.dumps({
        "name": historical_reference.name,
        "recorded_score": historical_reference.validation_top1,
        "seconds": historical_reference.seconds,
        "source": historical_reference.source,
        "directly_comparable": False,
    }, indent=2, default=str))

recipe = REPOSITORY_RECIPES.get((DATASET, MODEL))
if recipe is None:
    print("No registered same-protocol reference recipe exists for this model and dataset.")
else:
    configured_names = {spec.name for spec in config.search_space}
    reference_params = {
        name: value
        for name, value in recipe.items()
        if name in configured_names and name not in {"epochs", "augmentation"}
    }
    if "strong_augmentation" in configured_names and "augmentation" in recipe:
        reference_params["strong_augmentation"] = recipe["augmentation"] == "strong"
    if "channels_last" in configured_names:
        reference_params["channels_last"] = MODEL.startswith("resnet") and RESOLVED_DEVICE == "cuda"

    reference_batch = reference_params.get("batch_size")
    if reference_batch is not None and int(reference_batch) not in completed_batches:
        print(
            "The exact reference batch did not complete calibration on this hardware. "
            "The reference track is disabled rather than silently substituting another batch."
        )
    else:
        reference_name = f"{MODEL}-{DATASET}-full-reference-split{SPLIT_SEED}-s{SEARCH_SEED}-{RUN_LABEL}"
        REFERENCE_CONFIG_PATH = PERSIST_ROOT / f"{reference_name}.yaml"
        reference_base = copy.deepcopy(base)
        reference_base["study"].update({
            "name": reference_name,
            "output_dir": str(PERSIST_ROOT),
            "storage_path": str(PERSIST_ROOT / f"{reference_name}.db"),
            "seed": SEARCH_SEED,
            "resume": True,
        })
        reference_base["search"]["mode"] = "full"
        reference_base["search"]["sampler"] = "random"
        reference_base["continuous"]["enabled"] = False
        reference_base["objectives"] = [
            {"name": "validation_top1", "direction": "maximize", "primary": True}
        ]
        reference_base["constraints"] = []
        reference_base["search_space"] = {
            name: {"type": "fixed", "default": value}
            for name, value in reference_params.items()
        }
        reference_base["proxy"]["enabled"] = False
        reference_base["successive_halving"]["enabled"] = False
        reference_base["full"].update({
            "enabled": True,
            "trials": 1,
            "exhaustive": False,
            "maximum_combinations": 1,
            "allow_performance_pruning": False,
            "allow_safety_termination": True,
            "evaluate_test": False,
        })
        REFERENCE_CONFIG_PATH.write_text(
            yaml.safe_dump(reference_base, sort_keys=False), encoding="utf-8"
        )
        reference_config = load_study_config(REFERENCE_CONFIG_PATH)
        print("Same-protocol reference configuration:")
        print(json.dumps({
            "study_name": reference_name,
            "params": reference_params,
            "epochs": reference_config.full.budget.epochs,
            "seeds": reference_config.full.budget.seeds,
            "split_seed": reference_config.dataset.get("split_seed"),
            "evaluate_test": reference_config.full.evaluate_test,
        }, indent=2, default=str))

        if RUN_REFERENCE_TRACK:
            print(json.dumps(HpoStudy(REFERENCE_CONFIG_PATH, repo_root=REPO_ROOT).run(), indent=2, default=str))
        else:
            print("Reference track not started. Review its configuration before enabling it.")

        discovery_path = PERSIST_ROOT / STUDY_NAME / "trials.jsonl"
        reference_path = PERSIST_ROOT / reference_name / "trials.jsonl"
        discovery_rows = read_jsonl(discovery_path) if discovery_path.exists() else []
        reference_rows = read_jsonl(reference_path) if reference_path.exists() else []
        discovery_full = [row for row in discovery_rows if row.get("status") == "completed" and row.get("stage") == "full"]
        reference_full = [row for row in reference_rows if row.get("status") == "completed" and row.get("stage") == "full"]

        if discovery_full and reference_full:
            best_discovered = max(discovery_full, key=lambda row: row["metrics"].get("validation_top1", float("-inf")))
            best_reference = max(reference_full, key=lambda row: row["metrics"].get("validation_top1", float("-inf")))
            fair_comparison = {
                "comparison_type": "same-protocol validation comparison",
                "directly_comparable": True,
                "discovered": best_discovered,
                "reference": best_reference,
                "validation_accuracy_gap": (
                    best_discovered["metrics"].get("validation_top1")
                    - best_reference["metrics"].get("validation_top1")
                ),
                "parameter_distance": normalized_parameter_distance(
                    best_discovered.get("params", {}), reference_params
                ),
            }
            FAIR_COMPARISON_PATH = PERSIST_ROOT / f"{STUDY_NAME}-fair-reference-comparison.json"
            FAIR_COMPARISON_PATH.write_text(
                json.dumps(fair_comparison, indent=2, default=str), encoding="utf-8"
            )
            print(json.dumps(fair_comparison, indent=2, default=str))
        else:
            print("A fair comparison requires completed full-fidelity discovery and reference results.")


## 16. Export selected hyperparameters and study artifacts


This cell creates a clean export directory and ZIP file. The most important file for the next project step is `selected_hyperparameters.json`. It records the selected candidate, its metrics, the hardware context, and the exact parameters I can pass into the final training notebook.


In [ ]:
EXPORT = True

selected_candidate = globals().get("selected_candidate")
REFERENCE_CONFIG_PATH = globals().get("REFERENCE_CONFIG_PATH")
FAIR_COMPARISON_PATH = globals().get("FAIR_COMPARISON_PATH")

if EXPORT and (PERSIST_ROOT / STUDY_NAME).exists():
    study_dir = PERSIST_ROOT / STUDY_NAME
    export_dir = PERSIST_ROOT / f"{STUDY_NAME}-export"
    if export_dir.exists():
        shutil.rmtree(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)

    # Copy the report artifacts produced by the current framework.
    allowed_suffixes = {".csv", ".json", ".png", ".log"}
    for source in study_dir.iterdir():
        if source.is_file() and source.suffix.lower() in allowed_suffixes:
            shutil.copy2(source, export_dir / source.name)

    shutil.copy2(CONFIG_PATH, export_dir / CONFIG_PATH.name)
    shutil.copy2(ESTIMATE_PATH, export_dir / ESTIMATE_PATH.name)
    database_path = PERSIST_ROOT / f"{STUDY_NAME}.db"
    if database_path.exists():
        shutil.copy2(database_path, export_dir / database_path.name)
    if REFERENCE_CONFIG_PATH is not None and REFERENCE_CONFIG_PATH.exists():
        shutil.copy2(REFERENCE_CONFIG_PATH, export_dir / REFERENCE_CONFIG_PATH.name)
    if FAIR_COMPARISON_PATH is not None and FAIR_COMPARISON_PATH.exists():
        shutil.copy2(FAIR_COMPARISON_PATH, export_dir / FAIR_COMPARISON_PATH.name)

    if selected_candidate is not None:
        selected_payload = {
            "study_name": STUDY_NAME,
            "selection_rule": "Pareto knee when available; otherwise highest final validation accuracy",
            "dataset": DATASET,
            "model": MODEL,
            "validation_fraction": VALIDATION_FRACTION,
            "split_seed": SPLIT_SEED,
            "search_seed": SEARCH_SEED,
            "hardware": hardware.to_dict(),
            "runtime": asdict(config.runtime),
            "params": selected_candidate.get("params", {}),
            "metrics": selected_candidate.get("metrics", {}),
            "stage": selected_candidate.get("stage"),
            "test_set_used_for_selection": False,
        }
        (export_dir / "selected_hyperparameters.json").write_text(
            json.dumps(selected_payload, indent=2, default=str), encoding="utf-8"
        )

    notebook_metadata = {
        "study_name": STUDY_NAME,
        "repository_revision": revision,
        "mode": MODE,
        "continuous": CONTINUOUS,
        "objective_profile": OBJECTIVE_PROFILE,
        "effective_sampler": EFFECTIVE_SAMPLER,
        "search_seed": SEARCH_SEED,
        "split_seed": SPLIT_SEED,
        "cost_estimation_available": COST_ESTIMATION_AVAILABLE,
        "missing_cost_rates": MISSING_COST_RATES,
        "historical_baseline_directly_comparable": False,
        "same_protocol_reference_available": bool(
            FAIR_COMPARISON_PATH is not None and FAIR_COMPARISON_PATH.exists()
        ),
    }
    (export_dir / "notebook_experiment_metadata.json").write_text(
        json.dumps(notebook_metadata, indent=2, default=str), encoding="utf-8"
    )

    archive = shutil.make_archive(str(export_dir), "zip", root_dir=export_dir)
    print("Export directory:", export_dir)
    print("Export archive:", archive)
else:
    print("Nothing to export yet. Run or resume a study first.")


## 17. Common errors and what they mean

### `CUDA unavailable`
The runtime is using CPU. In Colab, select `Runtime → Change runtime type → GPU`, reconnect, and rerun the notebook from Setup.

### `No candidate batch completed calibration`
Every proposed batch failed or left insufficient memory headroom. Reduce the batch choices, disable compilation, reduce the model size, use gradient accumulation, or increase the memory reserve.

### `persisted study configuration hash does not match`
The study name already exists, but the current configuration is different. Restore the original settings to resume, or change `RUN_LABEL` to begin a new study.

### `SQLite database is locked`
Another process may still be writing to the study. Stop duplicate notebook executions, wait briefly, and resume with only one controller process.

### `loss is NaN` or a trial is marked divergent
The learning rate or another optimizer setting is unstable. The framework records the failure and should continue with other candidates. Review whether the search range includes unreasonable values.

### `out of memory`
The batch, architecture, or concurrent workload exceeds available VRAM. Use the calibrated batch choices, increase the reserve, or reduce concurrency.

### ImageNet-32 files are missing
The current loader expects the prepared `.npy` files and `stats.json` under `src/a1-cv/data/imagenet32`. A different path in the notebook alone does not change the subprocess loader.

### Cost is unavailable
GPU-hours and CPU-hours are still recorded, but dollar cost requires the user-supplied marginal GPU, CPU, and storage rates. I do not hard-code Colab prices because they vary by plan, hardware, and time.

### The reference benchmark does not run
The exact reference batch may not fit the assigned hardware, or the repository may not contain a registered recipe for the selected model and dataset. The notebook does not silently substitute a different reference configuration.


## 18. Potential improvements

The current framework is useful and testable, but these are the improvements I would prioritize next:

1. **Validate estimator accuracy with holdout trials.** The notebook now uses an end-to-end pilot, but the strongest validation would compare predicted and measured times across several batches, architectures, and precision modes on the same Colab GPU.
2. **Separate one-time and per-epoch overhead inside the framework.** Compilation, first CUDA initialization, validation, checkpointing, and FLOP counting should be modeled independently instead of grouping all non-training pilot time together.
3. **Add measured multi-GPU efficiency.** The framework can schedule one independent trial per GPU, but the parallel makespan should be calibrated on actual multi-GPU hardware before making strong scaling claims.
4. **Add automatic repeated-seed confirmation.** After selecting a Pareto candidate, the framework could automatically run several seeds and report mean, standard deviation, and confidence intervals.
5. **Execute the smoke notebook in CI.** A bounded CPU workflow could validate notebook syntax, configurations, resume behavior, and exports on every pull request.
6. **Improve ImageNet-32 path configuration.** The adapter should honor a dataset root from the study configuration instead of relying on the module-level default path.
7. **Add audio-specific accounting for the next project.** The same estimator should count audio hours, padded sequence length, feature frames, and decoding cost when we adapt the framework to a speech transformer.


## 19. Final-project handoff

After completing the search, you can export `selected_hyperparameters.json` into the final report notebook. That notebook should:

1. Load the selected ResNet and ViT parameters.
2. Retrain the selected candidates using the full dataset and confirmation seeds.
3. Evaluate the final test set only after model selection is complete.
4. Compare accuracy, runtime, GPU-hours, CPU-hours, memory, data consumed, and cost.
5. Include the Pareto plots and explain why the selected candidate is a reasonable trade-off.
6. Summarize how the framework changes its execution plan for different hardware.
7. Separate measured results from projections and historical repository results.

For the future Bambara/Dyula audio-language-transformer project, I can keep the HPO study, search modes, persistence, Pareto analysis, and reporting. I would replace the vision adapters with an audio dataset adapter and a speech-model adapter, then tune parameters such as the encoder, frozen layers, clip duration, batch audio seconds, learning rates, masking, dropout, gradient accumulation, and decoding settings. The resource estimator would also need to track audio hours and sequence lengths instead of only image counts.
